# NeUF analysis workbench

## Task index

1. Phase 1 E2 alpha=2 diagnostic
2. Full-dataset NeUF/UltraNeRF comparison with GT-only contrast ROIs
3. Phase 1 checkpoint native-resolution montages

## 1. Phase 1 E2 alpha=2 diagnostic

Question: what does the extrapolated output `anatomy + 2 * speckle` look like on the fixed step-20000 validation slices? This is diagnostic only; the production model contract remains `alpha in [0, 1]`.

In [ ]:
from pathlib import Path
import csv

import matplotlib.pyplot as plt
import numpy as np
import torch

workspace = Path.cwd().resolve()
input_dir = workspace / 'experiments/20260901_trial02/phase1_e1_plain_dual_vs_e2_hf02/E2/seed3407/images'
output_root = workspace / 'logs/20260901_train02/cerebral_patient0/index_all/E2_alpha2_diagnostic'
plot_dir = output_root / 'plots/step_020000'
metric_dir = output_root / 'metrics'
plot_dir.mkdir(parents=True, exist_ok=True)
metric_dir.mkdir(parents=True, exist_ok=True)
tensor_paths = sorted(input_dir.glob('validation_020000_*.pt'))
assert len(tensor_paths) == 4, f'Expected four fixed validation tensors, found {len(tensor_paths)}'
tensor_paths

In [ ]:
records = []
for path in tensor_paths:
    payload = torch.load(path, map_location='cpu', weights_only=False)
    required = {'slice_id', 'target', 'anatomy', 'speckle'}
    assert required <= payload.keys(), f'{path.name} is missing {required - payload.keys()}'
    target = payload['target'].detach().cpu().float().squeeze().numpy()
    anatomy = payload['anatomy'].detach().cpu().float().squeeze().numpy()
    speckle = payload['speckle'].detach().cpu().float().squeeze().numpy()
    assert target.shape == anatomy.shape == speckle.shape
    alpha1 = anatomy + speckle
    alpha2 = anatomy + 2.0 * speckle
    records.append({
        'slice_id': str(payload['slice_id']),
        'target': target,
        'anatomy': anatomy,
        'alpha1': alpha1,
        'alpha2': alpha2,
        'alpha2_min': float(alpha2.min()),
        'alpha2_max': float(alpha2.max()),
        'below_zero_fraction': float(np.mean(alpha2 < 0.0)),
        'above_one_fraction': float(np.mean(alpha2 > 1.0)),
        'mae_alpha2': float(np.mean(np.abs(alpha2 - target))),
    })
[(item['slice_id'], item['alpha2_min'], item['alpha2_max']) for item in records]

In [ ]:
figure, axes = plt.subplots(len(records), 5, figsize=(16, 3.2 * len(records)), squeeze=False)
for row, item in enumerate(records):
    panels = [
        ('GT', item['target'], 'gray', 0.0, 1.0),
        ('alpha=0 (anatomy)', item['anatomy'], 'gray', 0.0, 1.0),
        ('alpha=1', item['alpha1'], 'gray', 0.0, 1.0),
        ('alpha=2', item['alpha2'], 'gray', 0.0, 1.0),
        ('|alpha=2 - GT|', np.abs(item['alpha2'] - item['target']), 'magma', 0.0, 0.5),
    ]
    for axis, (title, image, cmap, vmin, vmax) in zip(axes[row], panels):
        axis.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
        axis.set_title(title)
        axis.axis('off')
    axes[row, 0].set_ylabel(item['slice_id'])
figure.suptitle('E2 step 20000: fixed validation slices (shared display range [0, 1])', y=1.0)
figure.tight_layout()
figure_path = plot_dir / 'alpha_0_1_2_comparison.png'
figure.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
figure_path

In [ ]:
metric_path = metric_dir / 'alpha2_range_stats.csv'
fields = ['slice_id', 'alpha2_min', 'alpha2_max', 'below_zero_fraction', 'above_one_fraction', 'mae_alpha2']
with metric_path.open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=fields)
    writer.writeheader()
    writer.writerows([{key: item[key] for key in fields} for item in records])
summary = {
    'slice_count': len(records),
    'mean_below_zero_fraction': float(np.mean([item['below_zero_fraction'] for item in records])),
    'mean_above_one_fraction': float(np.mean([item['above_one_fraction'] for item in records])),
    'mean_mae_alpha2': float(np.mean([item['mae_alpha2'] for item in records])),
}
summary, metric_path

## 2. Full-dataset NeUF/UltraNeRF comparison

Evaluate NeUF E0/E1/E2 and UltraNeRF Hash on all 242 frames in one common convex render grid. MSE, SSIM, LPIPS, and gCNR use one shared mask. Contrast ROI pairs are selected deterministically from GT only; model predictions never influence ROI selection.

In [ ]:
from neuf.full_dataset_comparison import ComparisonConfig, run_comparison

workspace_root = Path('/home/zchen/Code').resolve()
comparison_output = workspace_root / 'logs/20260901_train01/cerebral_patient0/index_all/cross_model_comparison'
comparison_config = ComparisonConfig(
    workspace_root=workspace_root,
    dataset_path=workspace_root / 'NeUF/data/cerebral_data/Pre_traitement_echo_v2/Recalage/Patient0/us_recal_original/baked_dataset_physical.pkl',
    ultra_dataset_dir=workspace_root / 'UltraNeRF-Studio/data/cerebral/patient_0_convex',
    ultra_config_path=workspace_root / 'UltraNeRF-Studio/configs/config_cerebral_patient0_hash.txt',
    checkpoints={
        'NeUF_E0': workspace_root / 'NeUF/experiments/20260901_trial01/phase1_e1_plain_dual_vs_e2_hf02/E0/seed3407/checkpoints/ckpt_20000.pkl',
        'NeUF_E1': workspace_root / 'NeUF/experiments/20260901_trial02/phase1_e1_plain_dual_vs_e2_hf02/E1/seed3407/checkpoints/ckpt_20000.pkl',
        'NeUF_E2': workspace_root / 'NeUF/experiments/20260901_trial02/phase1_e1_plain_dual_vs_e2_hf02/E2/seed3407/checkpoints/ckpt_20000.pkl',
        'UltraNeRF_Hash': workspace_root / 'UltraNeRF-Studio/logs/20260828_train03/cerebral_patient0/index_all/ultranerf_hash/030000.tar',
    },
    output_dir=comparison_output,
    base_seed=20260901,
    roi_size=24,
    roi_pairs_per_frame=3,
    histogram_bins=64,
)
run_comparison(comparison_config)

## 3. Phase 1 checkpoint native-resolution montages

Question and acceptance criterion: export every saved E0/E1/E2 checkpoint preview at exactly the GT tensor's native pixel dimensions, then assemble lossless overview montages without resampling any panel. The fixed four validation slices, display ranges, orientation, and ordering are shared across models and steps.

Inputs are the step-matched validation tensors saved during training for checkpoints 5000, 10000, 15000, and 20000. Existing checkpoints, tensors, and preview PNGs remain read-only.

In [ ]:
from pathlib import Path
import csv

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont

workspace = Path.cwd().resolve()
experiment_roots = {
    'E0': workspace / 'experiments/20260901_trial01/phase1_e1_plain_dual_vs_e2_hf02/E0/seed3407',
    'E1': workspace / 'experiments/20260901_trial02/phase1_e1_plain_dual_vs_e2_hf02/E1/seed3407',
    'E2': workspace / 'experiments/20260901_trial02/phase1_e1_plain_dual_vs_e2_hf02/E2/seed3407',
}
steps = (5000, 10000, 15000, 20000)
output_root = workspace / 'logs/20260902_train01/cerebral_patient0/index_all/phase1_ckpt_native_montage'
plot_dir = output_root / 'plots'
individual_dir = plot_dir / 'individual'
metric_dir = output_root / 'metrics'
individual_dir.mkdir(parents=True, exist_ok=True)
metric_dir.mkdir(parents=True, exist_ok=True)

records = {}
reference_targets = {}
slice_ids = None
for model, root in experiment_roots.items():
    for step in steps:
        checkpoint = root / 'checkpoints' / f'ckpt_{step}.pkl'
        assert checkpoint.is_file(), f'Missing checkpoint: {checkpoint}'
        tensor_paths = sorted((root / 'images').glob(f'validation_{step:06d}_*.pt'))
        assert len(tensor_paths) == 4, f'{model} step {step}: expected four tensors, found {len(tensor_paths)}'
        current_ids = []
        for tensor_path in tensor_paths:
            payload = torch.load(tensor_path, map_location='cpu', weights_only=False)
            required = {'iteration', 'slice_id', 'target', 'prediction'}
            assert required <= payload.keys(), f'{tensor_path.name} is missing {required - payload.keys()}'
            assert int(payload['iteration']) == step
            item = {
                key: payload[key].detach().cpu().float().squeeze().numpy()
                for key in ('target', 'prediction')
            }
            assert item['target'].ndim == 2 and item['target'].shape == item['prediction'].shape
            if model == 'E2':
                for key in ('anatomy', 'speckle'):
                    assert key in payload, f'{tensor_path.name} is missing {key}'
                    item[key] = payload[key].detach().cpu().float().squeeze().numpy()
                    assert item[key].shape == item['target'].shape
            item['slice_id'] = str(payload['slice_id'])
            item['source_tensor'] = tensor_path
            current_ids.append(item['slice_id'])
            previous_target = reference_targets.setdefault(item['slice_id'], item['target'])
            assert np.array_equal(previous_target, item['target']), f"GT changed for {item['slice_id']}"
            records[(model, step, item['slice_id'])] = item
        if slice_ids is None:
            slice_ids = current_ids
        assert current_ids == slice_ids, f'{model} step {step}: validation slice order changed'

assert slice_ids is not None
native_shape = reference_targets[slice_ids[0]].shape
assert all(image.shape == native_shape for image in reference_targets.values())
{'checkpoint_count': len(experiment_roots) * len(steps), 'slice_ids': slice_ids, 'native_shape_hw': native_shape}

In [ ]:
def grayscale_image(array, upper=1.0):
    values = np.clip(np.asarray(array, dtype=np.float32) / upper, 0.0, 1.0)
    return Image.fromarray(np.rint(values * 255.0).astype(np.uint8), mode='L')


def colour_image(array, colour_map, minimum, maximum):
    normalized = np.clip((np.asarray(array, dtype=np.float32) - minimum) / (maximum - minimum), 0.0, 1.0)
    rgb = np.rint(plt.get_cmap(colour_map)(normalized)[..., :3] * 255.0).astype(np.uint8)
    return Image.fromarray(rgb, mode='RGB')


def load_font(size):
    try:
        return ImageFont.truetype('DejaVuSans.ttf', size=size)
    except OSError:
        return ImageFont.load_default()


def save_montage(path, row_items):
    height, width = native_shape
    left_margin, top_margin, gutter = 250, 92, 8
    canvas = Image.new(
        'RGB',
        (left_margin + len(slice_ids) * width + (len(slice_ids) - 1) * gutter,
         top_margin + len(row_items) * height + (len(row_items) - 1) * gutter),
        'white',
    )
    draw = ImageDraw.Draw(canvas)
    header_font, row_font = load_font(22), load_font(28)
    for column, slice_id in enumerate(slice_ids):
        x = left_margin + column * (width + gutter)
        draw.multiline_text((x + 8, 12), slice_id.replace('_', '\n', 2), fill='black', font=header_font, spacing=2)
    for row, (label, images) in enumerate(row_items):
        y = top_margin + row * (height + gutter)
        draw.text((12, y + height // 2), label, fill='black', font=row_font, anchor='lm')
        for column, image in enumerate(images):
            assert image.size == (width, height), f'{label}: panel was resampled or has wrong dimensions'
            x = left_margin + column * (width + gutter)
            canvas.paste(image.convert('RGB'), (x, y))
    canvas.save(path, format='PNG', compress_level=6)
    return canvas.size


dimension_rows = []
for model in experiment_roots:
    for step in steps:
        step_dir = individual_dir / model / f'step_{step:06d}'
        step_dir.mkdir(parents=True, exist_ok=True)
        for slice_id in slice_ids:
            item = records[(model, step, slice_id)]
            images = {
                'gt': (grayscale_image(item['target']), '[0,1]'),
                'prediction': (grayscale_image(item['prediction']), '[0,1]'),
                'absolute_error': (colour_image(np.abs(item['prediction'] - item['target']), 'magma', 0.0, 0.5), '[0,0.5]'),
            }
            if model == 'E2':
                images['anatomy'] = (grayscale_image(item['anatomy']), '[0,1]')
                images['speckle'] = (colour_image(item['speckle'], 'coolwarm', -0.5, 0.5), '[-0.5,0.5]')
            for kind, (image, display_range) in images.items():
                destination = step_dir / f'{slice_id}_{kind}.png'
                image.save(destination, format='PNG', compress_level=6)
                dimension_rows.append({
                    'artifact': str(destination.relative_to(output_root)),
                    'kind': kind,
                    'model': model,
                    'step': step,
                    'slice_id': slice_id,
                    'source_tensor': str(item['source_tensor']),
                    'width': image.width,
                    'height': image.height,
                    'display_range': display_range,
                })

prediction_rows = [('GT', [grayscale_image(reference_targets[slice_id]) for slice_id in slice_ids])]
error_rows = []
for model in experiment_roots:
    for step in steps:
        label = f'{model} step {step}'
        prediction_rows.append((label, [grayscale_image(records[(model, step, slice_id)]['prediction']) for slice_id in slice_ids]))
        error_rows.append((label, [colour_image(np.abs(records[(model, step, slice_id)]['prediction'] - reference_targets[slice_id]), 'magma', 0.0, 0.5) for slice_id in slice_ids]))

component_rows = []
for step in steps:
    component_rows.append((f'E2 anatomy {step}', [grayscale_image(records[('E2', step, slice_id)]['anatomy']) for slice_id in slice_ids]))
    component_rows.append((f'E2 speckle {step}', [colour_image(records[('E2', step, slice_id)]['speckle'], 'coolwarm', -0.5, 0.5) for slice_id in slice_ids]))

montage_paths = {
    'predictions': plot_dir / 'all_checkpoints_predictions.png',
    'absolute_errors': plot_dir / 'all_checkpoints_absolute_errors.png',
    'e2_components': plot_dir / 'e2_all_checkpoints_components.png',
}
montage_sizes = {
    'predictions': save_montage(montage_paths['predictions'], prediction_rows),
    'absolute_errors': save_montage(montage_paths['absolute_errors'], error_rows),
    'e2_components': save_montage(montage_paths['e2_components'], component_rows),
}
montage_sizes

In [ ]:
dimension_path = metric_dir / 'image_dimensions.csv'
with dimension_path.open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(dimension_rows[0]))
    writer.writeheader()
    writer.writerows(dimension_rows)

native_height, native_width = native_shape
assert len(dimension_rows) == 176
assert all((row['width'], row['height']) == (native_width, native_height) for row in dimension_rows)
for path in montage_paths.values():
    assert path.is_file() and path.stat().st_size > 0

for name, path in montage_paths.items():
    with Image.open(path) as full_image:
        assert full_image.size == montage_sizes[name]
        preview = full_image.copy()
    preview.thumbnail((1600, 1600), Image.Resampling.LANCZOS)
    print(f'{name}: full={montage_sizes[name]}, inline_preview={preview.size}, path={path}')
    display(preview)

{
    'status': 'complete',
    'checkpoint_count': 12,
    'slice_count': len(slice_ids),
    'native_panel_wh': (native_width, native_height),
    'individual_png_count': len(dimension_rows),
    'dimension_manifest': dimension_path,
}